<a href="https://colab.research.google.com/github/Rishii077/AI-Lab-Assignments/blob/main/Experiment_4_ReAct_SQL_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 17.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.1 which is incompatible.


In [2]:
from google.colab import userdata
from google import genai

API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

print("Gemini connected successfully!")

Gemini connected successfully!


In [3]:
import sqlite3

conn = sqlite3.connect("college.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    marks INTEGER
)
""")

cursor.execute("DELETE FROM students")

students = [
    (1, "Rahul", "CSE", 85),
    (2, "Priya", "ECE", 91),
    (3, "Arjun", "CSE", 78),
    (4, "Sneha", "IT", 88),
    (5, "Kiran", "ECE", 95)
]

cursor.executemany(
    "INSERT INTO students VALUES (?, ?, ?, ?)",
    students
)

conn.commit()

print("Database created successfully!")

Database created successfully!


In [4]:
def database_tool(sql_query):

    try:
        cursor.execute(sql_query)
        results = cursor.fetchall()
        return results

    except Exception as e:
        return f"Database Error: {e}"

print("Database tool created successfully!")

Database tool created successfully!


In [5]:
def react_sql_agent(question):

    print("========== REACT SQL AGENT ==========")

    # -------- REASON --------
    reasoning_prompt = f"""
You are a ReAct SQL Agent.

You have access to a SQLite database containing this table:

students(
    id,
    name,
    department,
    marks
)

The user has asked:

{question}

Reason about what information is required and determine
what SQL query should be executed.

Return ONLY the SQL query.
Do not use markdown code blocks.
"""

    interaction = client.interactions.create(
        model="gemini-3.6-flash",
        input=reasoning_prompt
    )

    sql_query = interaction.output_text.strip()

    sql_query = sql_query.replace("```sql", "")
    sql_query = sql_query.replace("```", "")
    sql_query = sql_query.strip()

    print("\nTHOUGHT / PLAN:")
    print("Determine the required database information and generate SQL.")

    # -------- ACTION --------
    print("\nACTION:")
    print("Calling database tool...")

    print("\nSQL Query:")
    print(sql_query)

    tool_result = database_tool(sql_query)

    # -------- OBSERVATION --------
    print("\nOBSERVATION:")
    print(tool_result)

    # -------- FINAL ANSWER --------
    final_prompt = f"""
You are a helpful assistant.

User Question:
{question}

SQL Query:
{sql_query}

Database Tool Result:
{tool_result}

Provide a clear and simple final answer to the user.

Do not mention internal reasoning.
"""

    final_interaction = client.interactions.create(
        model="gemini-3.6-flash",
        input=final_prompt
    )

    final_answer = final_interaction.output_text.strip()

    print("\nFINAL ANSWER:")
    print(final_answer)

In [6]:
react_sql_agent(
    "Which students have marks greater than 90?"
)

========== REACT SQL AGENT ==========

THOUGHT / PLAN:
Determine the required database information and generate SQL.

ACTION:
Calling database tool...

SQL Query:
SELECT * FROM students WHERE marks > 90;

OBSERVATION:
[(2, 'Priya', 'ECE', 91), (5, 'Kiran', 'ECE', 95)]

FINAL ANSWER:
The students with marks greater than 90 are:

* **Priya** (Marks: 91)
* **Kiran** (Marks: 95)


In [7]:
react_sql_agent(
    "What is the average marks of all students?"
)

========== REACT SQL AGENT ==========

THOUGHT / PLAN:
Determine the required database information and generate SQL.

ACTION:
Calling database tool...

SQL Query:
SELECT AVG(marks) FROM students;

OBSERVATION:
[(87.4,)]

FINAL ANSWER:
The average mark of all students is 87.4.
